# Data Olympians CS Streaming Presentation Notebook

## Oliver's Notebook Workings


## 1. Setup

### 1.1 Import Libraries

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy import stats

### 1.2 Load the Dataset

In [3]:
# Load the dataset
df = pd.read_csv('netflix_titles.csv')

# Display the first few rows of the dataset
df.head()


## 2. Data Preparation

### 2.1 Missing Values

In [4]:
# Check for missing values
print(df.isnull().sum())

In [5]:
# Handle missing values
# For simplicity, let's fill missing values for `director` and `cast` with 'Unknown'
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)

In [6]:
# Fill missing values in `country` with 'Unknown'
df['country'].fillna('Unknown', inplace=True)

In [7]:
# For `date_added`, let's fill missing values with a placeholder or drop them if they are not critical
df['date_added'].fillna('Unknown', inplace=True)

In [8]:
# Format duration column, in order to carry out Calculation
def extract_number(duration):
    if isinstance(duration, str):
        return int(duration.split(' ')[0])
    else:
        return None

# Apply function to 'duration' column to create a new 'number' column
df['number'] = df['duration'].apply(extract_number)


In [9]:
# Now fill nulls in number column and duration column 
# Calculate mean for each type
mean_seasons = df[df['type'] == 'TV Show']['number'].mean()
mean_minutes = df[df['type'] == 'Movie']['number'].mean()

In [10]:
# Function goes through row by row, if its null and in number column, depending on type it fills with mean seasons or minutes
def fill_missing(row):
    if pd.isna(row['number']):
        if row['type'] == 'TV Show':
            return mean_seasons
        elif row['type'] == 'Movie':
            return mean_minutes
    else:
        return row['number']

In [11]:
# Apply function to fill missing values
df['number'] = df.apply(fill_missing, axis=1)

In [12]:
print(df.isnull().sum())

In [13]:
# check new column
df.head()

In [14]:
# Fill missing values for `rating` with 'Unknown' or the mode
df['rating'].fillna('Unknown', inplace=True)

In [15]:
# Verify missing values
print(df.isnull().sum())

### 2.2 Convert Data Types

In [16]:
# Convert `date_added` to datetime format
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# Ensure categorical columns are of type 'category'
df['type'] = df['type'].astype('category')
df['rating'] = df['rating'].astype('category')
df['listed_in'] = df['listed_in'].astype('category')

# Check data types
print(df.dtypes)


### 2.3 Standardise Column Names

In [17]:
# Standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Verify column names
print(df.columns)


### 2.4 Remove Duplicate Records

In [18]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Verify duplicates removal
print(df.duplicated().sum())


In [19]:
df['number'].describe()

### 2.5 Handle Outliers

In [20]:
# Handle Outliers, need to separate data between movies and series as an outlier for movie lenght is different to outlier for seasons
series_df = df[df['type'] == 'TV Show']
movies_df = df[df['type'] == 'Movie']


In [21]:
# Function to detect outliers using Z-score
def detect_outliers_zscore(data, threshold=3):
    mean = np.mean(data)
    std = np.std(data)
    z_scores = (data - mean) / std
    return np.abs(z_scores) > threshold

In [22]:
# Detect and remove outliers in series
series_outliers = detect_outliers_zscore(series_df['number'])
series_df = series_df[~series_outliers]

In [23]:
# Detect and remove outliers in movies
movies_outliers = detect_outliers_zscore(movies_df['number'])
movies_df = movies_df[~movies_outliers]


In [24]:
# Recombine the cleaned data
cleaned_df = pd.concat([series_df, movies_df])

In [25]:
cleaned_df.describe()

In [26]:
# convert df back to csv
cleaned_df.to_csv('cleaned_data.csv', index=True)


In [27]:
df.isnull().sum()

### 2.6 Verify the cleaned dataset

In [28]:
cleaned_df.info()

In [125]:
cleaned_df.sample(5)

## 3. Data Exploration
### 3.1 Content Type Distribution

In [29]:
type_distribution = df['type'].value_counts()
print(type_distribution)

In [30]:
# Plot Content Type Distribution
plt.figure(figsize=(8, 6))
type_distribution.plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Distribution of Content Types')
plt.xlabel('Type')
plt.ylabel('Number of Entries')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 3.2 Content Addition by Year and Type

In [115]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv('netflix_titles.csv')

# Convert 'date_added' to datetime format
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# Extract year from 'date_added'
df['year_added'] = df['date_added'].dt.year

# Group by year and type, then count the number of entries
content_by_year_and_type = df.groupby(['year_added', 'type']).size().reset_index(name='count')

# Display the first few rows of the prepared DataFrame
print(content_by_year_and_type.head())

# Plotting
plt.figure(figsize=(14, 8))

# Use seaborn to create a line plot
sns.lineplot(data=content_by_year_and_type, x='year_added', y='count', hue='type', marker='o')

plt.title('Content Addition by Year and Type')
plt.xlabel('Year')
plt.ylabel('Number of Shows Added')
plt.legend(title='Type')
plt.grid(True)

# Display the plot
plt.show()


### 3.3 Genre Distribution

In [32]:
# Split genres and count occurrences
genre_counts = df['listed_in'].str.split(',', expand=True).stack().value_counts()
print(genre_counts)

In [33]:
# Split genres and count occurrences
genre_counts = df['listed_in'].str.split(',', expand=True).stack().value_counts()

# Plot Genre Distribution
plt.figure(figsize=(12, 8))
genre_counts.head(10).plot(kind='bar', color='purple')
plt.title('Top 10 Genre Distribution')
plt.xlabel('Genre')
plt.ylabel('Number of Entries')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


### 3.4 Content Distribution by Country

In [34]:
country_distribution = df['country'].value_counts()
print(country_distribution)

In [35]:
# Compute country distribution
country_distribution = df['country'].value_counts()

# Plot Content Distribution by Country
plt.figure(figsize=(14, 8))
top_countries = country_distribution.head(10)  # Get top 10 countries
top_countries.plot(kind='bar', color='green')
plt.title('Top 10 Content Distribution by Country')
plt.xlabel('Country')
plt.ylabel('Number of Entries')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [36]:
# Timeseries Analysis

In [37]:
cleaned_df

In [38]:
# -- First time series Graph - Average Movie/season length over time --

# Preprocessing for timeseries on movies length/season 
movies_df['year_added'] = movies_df['date_added'].dt.year
series_df['year_added'] = series_df['date_added'].dt.year

In [39]:
# Split by type, groupby year added
movies_avg_length = movies_df.groupby('year_added')['number'].mean()
series_avg_seasons = series_df.groupby('year_added')['number'].mean()

In [40]:
# Final plot 
plt.figure(figsize=(14, 7))
plt.title('Average Movie/Season Length Over Time')
movies_avg_length.plot()
series_avg_seasons.plot()
plt.xlabel('Year')
plt.ylabel('Average Length (Minutes)')
plt.legend(['Movie','TV Show'])

In [41]:
# -- Second time seires - Counts of shows/movies added by country over time -- 

In [42]:
# Copy cleaned_df into new one to format for this plot
country_time_series = cleaned_df.copy()
country_time_series['year_added'] = country_time_series['date_added'].dt.year

In [43]:
# Preprocess new dataframe
# Split the 'country' column to separte out multiple countries on each row
country_time_series['country'] = country_time_series['country'].str.split(', ')


In [44]:
country_time_series

In [45]:
# Explode new data frame create new duplicate rows but with each different country
country_exploded = country_time_series.explode('country')

In [46]:
# Value counts to identify the largest 10 countries as the country is set too large to plot all of them
country_totals = country_exploded['country'].value_counts()

In [47]:
# Get top 10 countries
top_10_countries = country_totals.nlargest(10).index

In [48]:
# Get all the rows from the exploded data frame that feature these top 10 countries
df_top_10 = country_exploded[country_exploded['country'].isin(top_10_countries)]

In [49]:
df_top_10

In [50]:
country_counts_top_10 = df_top_10.groupby(['year_added', 'country']).size().unstack().fillna(0)

In [51]:
# Final Plot 
plt.figure(figsize=(12, 8))

# Plot each of the top 10 country's counts over time
for country in country_counts_top_10.columns:
    plt.plot(country_counts_top_10.index, country_counts_top_10[country], label=country)

plt.xlabel('Year')
plt.ylabel('Counts')
plt.title('Counts by Top 10 Countries Over Time')
plt.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

## Sabeeha's Notebook Workings

# Netflix Titles Dataset 

## 1. Setup

### 1.1 Import Libraries

In [52]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy import stats

### 1.2 Load the Dataset

In [53]:
df = pd.read_csv('cleaned_data.csv')
df.head()

## 2. Data Exploration

### Content Type Distribution

In [54]:
type_distribution = df['type'].value_counts()
print(type_distribution)

In [55]:
netflix_colours = ['#E50914', '#333333']
plt.figure(figsize=(8, 6))
type_distribution.plot(kind='bar', color=netflix_colours)
plt.title('Distribution of Content Types')
plt.xlabel('Type')
plt.ylabel('Number of Entries')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Content Addition by Year and Type

In [56]:
content_by_year_type = df.groupby(['release_year', 'type']).size().unstack().fillna(0)

custom_palette = {'Movie': 'red', 'TV Show': 'black'}
plt.figure(figsize=(12, 8))
sns.lineplot(data=content_by_year_type, palette=custom_palette, markers=True, dashes=False)
plt.title('Content Addition by Year and Type')
plt.xlabel('Year')
plt.ylabel('Number of Entries')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Content Type')
plt.show()

### Genre Distribution

In [57]:
genre_counts = df['listed_in'].str.split(',', expand=True).stack().value_counts()
print(genre_counts)

In [58]:
genre_counts = df['listed_in'].str.split(',', expand=True).stack().value_counts()
plt.figure(figsize=(12, 8))
netflix_colors = ['#E50914'] * 10 
genre_counts.head(10).plot(kind='bar', color=netflix_colors)
plt.title('Top 10 Genre Distribution')
plt.xlabel('Genre')
plt.ylabel('Number of Entries')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Content Distribution by Country

In [59]:
country_distribution = df['country'].value_counts()
plt.figure(figsize=(14, 8))
top_countries = country_distribution.head(10)
netflix_red = '#E50914'
top_countries.plot(kind='bar', color=netflix_red)

plt.title('Top 10 Content Distribution by Country')
plt.xlabel('Country')
plt.ylabel('Number of Entries')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Netflix Userbase Dataset 

## 1. Setup

### 1.1 Import Libraries

In [60]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy import stats

In [62]:
df = pd.read_csv('Netflix Userbase.csv')
df.head()

In [63]:
df['Join Date'] = pd.to_datetime(df['Join Date'], format='%d-%m-%y')
df['Last Payment Date'] = pd.to_datetime(df['Last Payment Date'], format='%d-%m-%y')

# Display the first few rows to verify the changes
df.head()

In [64]:
missing_values = df.isnull().sum()
print(missing_values)

In [65]:
df['Subscription Type'] = df['Subscription Type'].str.capitalize()
df['Country'] = df['Country'].str.title()
df['Device'] = df['Device'].str.title()
df['Gender'] = df['Gender'].str.capitalize()

df.head()

In [66]:
df['Subscription Type'] = df['Subscription Type'].str.capitalize()
df['Country'] = df['Country'].str.title()
df['Device'] = df['Device'].str.title()
df['Gender'] = df['Gender'].str.capitalize()
df['Device'] = df['Device'].replace({'Smart Tv': 'Smart TV'})
df.head()

In [67]:
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")
df = df.drop_duplicates()

print("\nData after removing duplicates:")
print(df.head())

## 2. Revenue Analysis
### a) Total Revenue

In [68]:
import pandas as pd
import matplotlib.pyplot as plt

revenue_by_subscription = df.groupby('Subscription Type')['Monthly Revenue'].sum()

total_revenue = df['Monthly Revenue'].sum()
print(f"Total Revenue: ${total_revenue}")

colors = ['#E50914', '#333333', '#CCCCCC']

revenue_by_subscription.plot(
    kind='pie', 
    autopct='%1.1f%%', 
    startangle=140, 
    title='Revenue Distribution by Subscription Type', 
    ylabel='', 
    colors=colors
)
plt.show()

### b) Average Revenue per User (ARPU)

In [69]:
arpu = df['Monthly Revenue'].mean()
print(f"Average Revenue per User (ARPU): ${arpu:.2f}")
revenues = df['Monthly Revenue']
plt.figure(figsize=(10, 6))
plt.hist(revenues, bins=20, color='#333333', edgecolor='black')  # Grey bars with black edges
plt.axvline(arpu, color='#E50914', linestyle='--', linewidth=2, label=f'ARPU: ${arpu:.2f}')  # Red line for ARPU
plt.title('Revenue Distribution with ARPU')
plt.xlabel('Monthly Revenue ($)')
plt.ylabel('Number of Users')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### c) Revenue by Subscription Type

In [70]:
revenue_by_subscription = df.groupby('Subscription Type')['Monthly Revenue'].sum()
print(revenue_by_subscription)

In [71]:
revenue_by_subscription = df.groupby('Subscription Type')['Monthly Revenue'].sum()
plt.figure(figsize=(10, 6))
revenue_by_subscription.plot(kind='bar', color=['#E50914', '#333333', '#CCCCCC'])  # Custom colors to match Netflix
plt.title('Total Revenue by Subscription Type')
plt.xlabel('Subscription Type')
plt.ylabel('Total Revenue ($)')
plt.xticks(rotation=45)
plt.show()

## 3. Churn Analysis
### a) Churn Rate Calculation

In [72]:
# Assume churn is defined as users who have not made a payment in the last month:

last_payment_cutoff = pd.Timestamp('2023-07-01')  # Example cutoff
churned_users = df[df['Last Payment Date'] < last_payment_cutoff]
churn_rate = len(churned_users) / len(df) * 100
print(f"Churn Rate: {churn_rate:.2f}%")

In [73]:
labels = ['Churned', 'Retained']
sizes = [churn_rate, 100 - churn_rate]
colors = ['#E50914', '#333333']  # Red for churned, Dark Grey for retained

plt.figure(figsize=(8, 6))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors)
plt.title('Churn Rate')
plt.show()


### b) Churn by Subscription Type

In [74]:
churn_by_subscription = churned_users['Subscription Type'].value_counts(normalize=True) * 100
print(churn_by_subscription)

In [75]:
churn_by_subscription.plot(kind='bar', color=['#E50914', '#333333', '#CCCCCC'], title='Churn by Subscription Type')
plt.ylabel('Percentage (%)')
plt.xlabel('Subscription Type')
plt.show()

### c) Churn by Device

In [76]:
churn_by_device = churned_users['Device'].value_counts(normalize=True) * 100
print(churn_by_device)

In [77]:
churn_by_device.plot(kind='bar', color=['#E50914', '#333333', '#CCCCCC', '#640402'], title='Churn by Device')
plt.ylabel('Percentage (%)')
plt.xlabel('Device')
plt.show()

## 4. Customer Segmentation
### a) Demographic Segmentation (Example: Age Groups)

In [78]:
age_bins = [25, 30, 35, 40, 45, 52]  # Adjusted bins with 46-51 in the final range
age_labels = ['26-30', '31-35', '36-40', '41-45', '46-51']  # Updated labels

df['Age Group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels, right=False)

demographic_segment = df.groupby(['Age Group', 'Gender'])['User ID'].count()
print(demographic_segment)

demographic_segment.unstack().plot(
    kind='bar', 
    stacked=True, 
    color=['#E50914', '#333333'], 
    title='User Demographics by Age Group and Gender'
)
plt.ylabel('Number of Users')
plt.xlabel('Age Group')
plt.show()


### b) Behavioral Segmentation

In [79]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

# Example DataFrame
# df = pd.read_csv('your_data_file.csv')  # Uncomment and adjust if loading data from a CSV

# Calculate the count of users by subscription type and device
behavioral_segment = df.groupby(['Subscription Type', 'Device'])['User ID'].count().unstack()

# Define a simple color map from yellow to red
cmap = mcolors.LinearSegmentedColormap.from_list(
    'yellow_to_red', 
    ['#ffffcc', '#ff0000']
)

# Plot the heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(behavioral_segment, annot=True, fmt="d", cmap=cmap, linewidths=0.5, linecolor='black')
plt.title('Behavioral Segmentation by Subscription Type and Device')
plt.ylabel('Subscription Type')
plt.xlabel('Device')
plt.show()


In [80]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

behavioral_segment = df.groupby(['Subscription Type', 'Device'])['User ID'].count().unstack()

cmap = mcolors.LinearSegmentedColormap.from_list(
    'yellow_to_red', 
    ['#ffffcc', '#ff0000']
)

plt.figure(figsize=(10, 6))
sns.heatmap(behavioral_segment, annot=True, fmt="d", cmap=cmap)
plt.title('Behavioral Segmentation by Subscription Type and Device')
plt.ylabel('Subscription Type')
plt.xlabel('Device')
plt.show()


## 5. Subscription Duration Analysis
### a) Average Subscription Duration

In [81]:
import matplotlib.pyplot as plt

# Calculate subscription duration in months
df['Subscription Duration'] = (df['Last Payment Date'] - df['Join Date']).dt.days / 30
average_duration = df['Subscription Duration'].mean()
print(f"Average Subscription Duration: {average_duration:.2f} months")

plt.figure(figsize=(10, 6))
plt.hist(df['Subscription Duration'], bins=10, color='#E50914', edgecolor='#333333')  # Netflix red and dark grey
plt.title('Distribution of Subscription Duration')
plt.xlabel('Duration (Months)')
plt.ylabel('Number of Users')
plt.show()

### b) Subscription Duration by Type

In [82]:
duration_by_subscription = df.groupby('Subscription Type')['Subscription Duration'].mean()
print(duration_by_subscription)

In [83]:
netflix_colors = ['#E50914', '#333333', '#FFFFFF']  # Red, Dark Grey, White

plt.figure(figsize=(10, 6))
sns.boxplot(x='Subscription Type', y='Subscription Duration', data=df, palette=netflix_colors)
plt.title('Subscription Duration by Type')
plt.xlabel('Subscription Type')
plt.ylabel('Duration (Months)')
plt.show()

## 6. Country-Level Insights
### a) Country-wise Revenue

In [84]:
revenue_by_country = df.groupby('Country')['Monthly Revenue'].sum()
print(revenue_by_country)

In [85]:
netflix_red = '#E50914'
netflix_black = '#333333'
plt.figure(figsize=(12, 8))
revenue_by_country.plot(kind='bar', color=netflix_red)
plt.title('Total Revenue by Country')
plt.xlabel('Country')
plt.ylabel('Revenue ($)')
plt.xticks(rotation=45)
plt.show()

### b) User Distribution by Country

In [86]:
users_by_country = df['Country'].value_counts(normalize=True) * 100
print(users_by_country)

In [87]:
colors = ['#E50914', '#B71C1C', '#F44336', '#9E9E9E', '#757575', '#616161', '#424242', '#212121']

plt.figure(figsize=(10, 8))
users_by_country.plot(kind='pie', autopct='%1.1f%%', startangle=140, colors=colors)
plt.title('User Distribution by Country')
plt.ylabel('')
plt.show()


### c) Subscription Type by Country

In [88]:
subscription_by_country = df.groupby(['Country', 'Subscription Type'])['User ID'].count()
print(subscription_by_country)

In [89]:
colors = ['#000000', '#555555', '#E50914']  # Black, Grey, Red

plt.figure(figsize=(12, 8))
subscription_by_country.unstack().plot(kind='bar', stacked=True, color=colors[:subscription_by_country.unstack().shape[1]])
plt.title('Subscription Type Distribution by Country')
plt.xlabel('Country')
plt.ylabel('Number of Users')
plt.xticks(rotation=45)
plt.legend(title='Subscription Type')
plt.show()


## Lorin's Notebook Workings



# 1. Setup
# 1.1 Import Libraries

In [90]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy import stats

# 1.2 Load the Dataset

In [91]:
# Load the dataset
df = pd.read_csv('cleaned_data.csv')


# 2. User Analysis 

# 2.1 Loading & Merging Relevant Datasets 

In [92]:
# Loading CSV with popularity data (source = )
pop = pd.read_csv('titles.csv')
pop.info()

In [93]:
# Merging with original df 
result = pd.merge(df, pop, on='title', how='outer') 
result.head()

# 2.2 IMDb Score & Genre Analysis 

In [94]:
# Counting genre entries 

# Define a function to count genres
def count_genre_keywords(genres):
    if isinstance(genres, str):
        genres = genres.lower()
        genre_counts = {
            'Drama': 0,
            'Comedy': 0,
            'Horror': 0,
            'Action': 0,
            'Thriller': 0,
            'Romance': 0,
            'Documentary': 0,
            'Sci-Fi': 0,
            'Fantasy': 0,
            'Animation': 0,
            'Other': 0
        }
        
        # Increment counts based on the presence of keywords
        if 'drama' in genres:
            genre_counts['Drama'] += 1
        if 'comedy' in genres:
            genre_counts['Comedy'] += 1
        if 'horror' in genres:
            genre_counts['Horror'] += 1
        if 'action' in genres:
            genre_counts['Action'] += 1
        if 'thriller' in genres:
            genre_counts['Thriller'] += 1
        if 'romance' in genres:
            genre_counts['Romance'] += 1
        if 'documentation' in genres:
            genre_counts['Documentary'] += 1
        if 'scifi' in genres or 'science fiction' in genres:
            genre_counts['Sci-Fi'] += 1
        if 'fantasy' in genres:
            genre_counts['Fantasy'] += 1
        if 'animation' in genres:
            genre_counts['Animation'] += 1
        
        # If none of the specified genres are found, increment 'Other'
        if sum(genre_counts.values()) == 0:
            genre_counts['Other'] += 1
        
        return genre_counts
    else:
        # Handle non-string or NaN cases by categorizing as 'Unknown'
        return {
            'Drama': 0,
            'Comedy': 0,
            'Horror': 0,
            'Action': 0,
            'Thriller': 0,
            'Romance': 0,
            'Documentary': 0,
            'Sci-Fi': 0,
            'Fantasy': 0,
            'Animation': 0,
            'Other': 1  # Count as 'Other' if genre is missing or not a string
        }

# Apply the function and aggregate the results
highly_rated_genres = result[result['imdb_score'] > 7]['genres'].apply(count_genre_keywords)

# Sum up the counts across all films
genre_totals = highly_rated_genres.apply(pd.Series).sum()

# Display the genre totals
genre_totals = genre_totals.sort_values(ascending=False)
print(genre_totals)



In [95]:
# Plotting genre distribution 
# Apply the function to count genres across all films
all_genres_counts = result['genres'].apply(count_genre_keywords)

# Convert the counts into a DataFrame and sum them up
genre_totals_all = pd.DataFrame(all_genres_counts.tolist()).sum()
genre_totals_all = genre_totals_all.sort_values(ascending=False)

# Plot the genre totals
plt.figure(figsize=(12, 8))
genre_totals_all.plot(kind='bar', color='red')

# Customize the plot
plt.title('Overall Genre Distribution')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()  # Adjust layout to prevent clipping

# Show the plot
plt.show()

In [96]:
# Plotting genre distribution for highly rated films 

# Plot the genre totals in red
plt.figure(figsize=(12, 8))
genre_totals.plot(kind='bar', color='red')

# Customize the plot
plt.title('Genre Distribution for Highly Rated Films')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()  # Adjust layout to prevent clipping

# Show the plot
plt.show()

In [97]:
# plotting mean IMDb score
def mean_imdb_score_for_genre(keyword):
    # Filter films that contain the genre keyword
    filtered = result[result['genres'].str.contains(keyword, case=False, na=False)]
    # Return the mean IMDb score for the filtered films
    return filtered['imdb_score'].mean()

# Rename and capitalize genre names in the DataFrame
result['genres'] = result['genres'].replace(
    {'documentation': 'Documentary', 'scifi': 'Sci-fi'}, regex=True
).str.capitalize()

# Define a list of genre keywords with capitalized names
genre_keywords = [
    'Drama', 'Comedy', 'Horror', 'Action', 'Thriller', 
    'Romance', 'Documentary', 'Sci-fi', 'Fantasy', 'Animation'
]

# Calculate the mean IMDb score for each genre keyword
mean_scores = {keyword: mean_imdb_score_for_genre(keyword) for keyword in genre_keywords}

# Convert the results into a DataFrame for easier plotting
mean_scores_df = pd.DataFrame(list(mean_scores.items()), columns=['Genre', 'Mean IMDb Score'])

# Sort the DataFrame by mean IMDb score in descending order
mean_scores_df = mean_scores_df.sort_values(by='Mean IMDb Score', ascending=False)

# Plot the mean IMDb scores
plt.figure(figsize=(12, 8))
plt.bar(mean_scores_df['Genre'], mean_scores_df['Mean IMDb Score'], color='red')

# Customize the plot
plt.title('Mean IMDb Score for Each Genre')
plt.xlabel('Genre')
plt.ylabel('Mean IMDb Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()  # Adjust layout to prevent clipping
plt.ylim(6, 7.2)  # Adjust y-axis limit to focus on the relevant range

# Show the plot
plt.show()

In [98]:
# Correct genre names in the DataFrame
def update_genre_names(genre_str):
    if isinstance(genre_str, str):
        genre_str = genre_str.lower()
        genre_str = genre_str.replace('documentation', 'documentary')
        genre_str = genre_str.replace('scifi', 'sci-fi')
    return genre_str

# Apply the function to update genre names
result['genres'] = result['genres'].apply(update_genre_names)

# Define a function to calculate counts for each genre keyword
def count_genre_occurrences(genres):
    genre_counts = {
        'Drama': 0,
        'Comedy': 0,
        'Horror': 0,
        'Action': 0,
        'Thriller': 0,
        'Romance': 0,
        'Documentary': 0,  # Corrected from 'Documentation'
        'Sci-Fi': 0,       # Corrected from 'SciFi'
        'Fantasy': 0,
        'Animation': 0,
        'Other': 0         # Include 'Other'
    }
    if isinstance(genres, str):
        genres = genres.lower()
        found_genre = False
        for genre in genre_counts.keys():
            if genre.lower() in genres:
                genre_counts[genre] += 1
                found_genre = True
        if not found_genre:
            genre_counts['Other'] += 1
    return genre_counts

# Apply the function to count genres
genre_counts_all = result['genres'].apply(count_genre_occurrences)

# Convert the counts into a DataFrame and sum them up
total_genre_counts = pd.DataFrame(genre_counts_all.tolist()).sum()

# Calculate high IMDb score counts (score >= 7)
high_score_counts = result[result['imdb_score'] >= 7]['genres'].apply(count_genre_occurrences)
high_score_counts = pd.DataFrame(high_score_counts.tolist()).sum()

# Calculate percentage of high IMDb score counts
percentage_high_score = (high_score_counts / total_genre_counts) * 100

# Sort the percentages in descending order
percentage_high_score = percentage_high_score.sort_values(ascending=False)

# Plot the percentages
plt.figure(figsize=(12, 8))
plt.bar(percentage_high_score.index, percentage_high_score, color='red')

# Customize the plot
plt.title('Percentage of Films with IMDb Score > 7 for Each Genre')
plt.xlabel('Genre')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()  # Adjust layout to prevent clipping

# Show the plot
plt.show()

# 2.3 IMDb Score & Release Year 

In [120]:
# Ensure 'release_year' is in the correct format (int or str)
# Use .loc to avoid SettingWithCopyWarning
result.loc[:,'release_year'] = pd.to_numeric(result['release_year_x'], errors='coerce').astype('Int64')

# Drop rows where 'release_year' or 'imdb_score' is missing
result = result.dropna(subset=['release_year_x', 'imdb_score'])

# Group by 'release_year' and calculate the mean IMDb score for each year
average_scores_by_year = result.groupby('release_year_x')['imdb_score'].mean()

# Plot the average IMDb scores by release year
plt.figure(figsize=(14, 8))
plt.plot(average_scores_by_year.index, average_scores_by_year, marker='o', linestyle='-', color='red')

# Customize the plot
plt.title('Average IMDb Score by Release Year')
plt.xlabel('Release Year')
plt.ylabel('Average IMDb Score')
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping

# Show the plot
plt.show()

In [121]:
import pandas as pd
import matplotlib.pyplot as plt

# Ensure 'release_year' is in the correct format (int or str)
result['release_year'] = pd.to_numeric(result['release_year_x'], errors='coerce').astype('Int64')

# Drop rows where 'release_year' is missing
result = result.dropna(subset=['release_year_x'])

# Count the number of films for each release year
film_counts_by_year = result['release_year_x'].value_counts().sort_index()

# Plot the frequency of films by release year
plt.figure(figsize=(14, 8))
plt.bar(film_counts_by_year.index, film_counts_by_year, color='red')

# Customize the plot
plt.title('Frequency of Films by Release Year')
plt.xlabel('Release Year')
plt.ylabel('Number of Films')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y')
plt.tight_layout()  # Adjust layout to prevent clipping

# Show the plot
plt.show()

# 2.4 IMDb Score & Release Country

In [102]:
import pandas as pd
import matplotlib.pyplot as plt

# Sample data preparation
# Ensure 'IMDb_score' is in numeric format
result['imdb_score'] = pd.to_numeric(result['imdb_score'], errors='coerce')

# Drop rows where 'IMDb_score' is missing
result = result.dropna(subset=['country', 'imdb_score'])

# Split countries listed together into separate rows
result['country'] = result['country'].apply(lambda x: [country.strip() for country in x.split(',')] if isinstance(x, str) else [])
result = result.explode('country')

# Drop rows where 'country' or 'IMDb_score' is missing after splitting
result = result.dropna(subset=['country', 'imdb_score'])

# Group by 'country' and calculate the mean IMDb score for each country
mean_scores_by_country = result.groupby('country')['imdb_score'].mean()

# Sort the results in descending order and select the top 10
top_10_countries = mean_scores_by_country.sort_values(ascending=False).head(5)

# Plot the mean IMDb scores for the top 10 countries
plt.figure(figsize=(14, 8))
plt.bar(top_10_countries.index, top_10_countries, color='red')

# Customize the plot
plt.title('Top 5 Countries by Mean IMDb Score')
plt.xlabel('Country')
plt.ylabel('Mean IMDb Score')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 10)  # IMDb scores range from 0 to 10
plt.grid(axis='y')
plt.tight_layout()  # Adjust layout to prevent clipping

# Show the plot
plt.show()

In [103]:
result['country'] = result['country'].apply(lambda x: [country.strip() for country in x.split(',')] if isinstance(x, str) else [])

# Explode the 'country' column so each country is in its own row
result = result.explode('country')

# Drop rows where 'country' is missing
result = result.dropna(subset=['country'])

# Count the number of Ukrainian films
ukrainian_film_count = result[result['country'].str.contains('Ukraine', case=False, na=False)].shape[0]

print(f'Number of Ukrainian films: {ukrainian_film_count}')

In [104]:
result['imdb_score'] = pd.to_numeric(result['imdb_score'], errors='coerce')

# Drop rows where 'imdb_score' is missing
result = result.dropna(subset=['country', 'imdb_score'])

# Split countries listed together into separate rows
result['country'] = result['country'].apply(lambda x: [country.strip() for country in x.split(',')] if isinstance(x, str) else [])
result = result.explode('country')

# Drop rows where 'country' or 'imdb_score' is missing after splitting
result = result.dropna(subset=['country', 'imdb_score'])

# Group by 'country' and calculate the mean IMDb score for each country
mean_scores_by_country = result.groupby('country')['imdb_score'].mean()

# Sort the results in ascending order and select the bottom 5
bottom_5_countries = mean_scores_by_country.sort_values(ascending=True).head(5)

# Plot the mean IMDb scores for the bottom 5 countries
plt.figure(figsize=(14, 8))
plt.bar(bottom_5_countries.index, bottom_5_countries, color='red')

# Customize the plot
plt.title('Bottom 5 Countries by Mean IMDb Score')
plt.xlabel('Country')
plt.ylabel('Mean IMDb Score')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 10)  # IMDb scores range from 0 to 10
plt.grid(axis='y')
plt.tight_layout()  # Adjust layout to prevent clipping

# Show the plot
plt.show()

# 3 Userbase Analysis

In [105]:
# Loading CSV with userbase data (source = )
userbase = pd.read_csv('Netflix Userbase.csv')

In [106]:
# Convert the 'join' and 'last_payment' columns to datetime objects
userbase['Join Date'] = pd.to_datetime(userbase['Join Date'], format='%d-%m-%y')
userbase['Last Payment Date'] = pd.to_datetime(userbase['Last Payment Date'], format='%d-%m-%y')

# Calculate the subscription length in days
userbase['Subscription Length'] = (userbase['Last Payment Date'] - userbase['Join Date']).dt.days

In [107]:
grouped_data = userbase.groupby('Country')['Subscription Type'].value_counts(normalize=True).unstack() * 100

colours = ['grey', 'red', 'black',]

# Plot the data as a stacked bar plot
plt.figure(figsize=(12, 8))
grouped_data.plot(kind='bar', stacked=True, ax=plt.gca(), color = colours)

# Customize the plot
plt.title('Subscription Distribution by Country (Percentage)')
plt.xlabel('Country')
plt.ylabel('Percentage')
plt.legend(title='Subscription Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')

# Show the plot
plt.tight_layout()  # Adjust layout to prevent clipping
plt.show()

In [108]:
mean_subscription_length = userbase.groupby('Country')['Subscription Length'].mean()

plt.figure(figsize=(12, 8))
mean_subscription_length.plot(kind='bar', color='red')

# Customize the plot
plt.title('Mean Subscription Length by Country')
plt.xlabel('Country')
plt.ylabel('Mean Subscription Length')
plt.xticks(rotation=45, ha='right')
plt.ylim(275, 330)
plt.tight_layout() 

In [109]:
userbase['Country'].nunique()


## Chris's Notebook Workings



# 1. Setup
# 1.1 Import Libraries

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy import stats
from bokeh.models import ColumnDataSource, Range1d, LinearAxis, HoverTool, LabelSet
from math import pi
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.transform import cumsum
from bokeh.palettes import Category20c

# 1.2 Load the Datasets

In [20]:
#I found that in the time provided these were the best CSVs to work with, for comparison with competeing streaming platforms.
df_min = pd.read_csv('MinuteSharing.csv')
df_libr = pd.read_csv('LibrarySize.csv')
df_movstr = pd.read_csv('moviestreams.csv')

# 2. Competing Platform  Analysis

# 2.1  Market Share by Minutes Data and Visuals

In [9]:
df_min

In [10]:
df_min.info()


In [21]:
#made a pie chart in matplotlib so show the market share by minutes.
plt.figure(figsize=(8, 8))

colors = ['#FF0000', '#FF3333', '#FF6666', '#FF9999', '#FFCCCC', '#D3D3D3']

plt.pie(df_min['Percentage of streaming'], labels = df_min['Streaming platform'], autopct='%1.1f%%', startangle = 140, colors = colors)

plt.title('Percentage of Streaming by Platform')

plt.savefig('Market_share_by_minutes.png', format='png', dpi=300)

plt.show()

In [12]:
total_percentage = df_min['Percentage of streaming'].sum()
if total_percentage != 100:
    raise ValueError(f"Total percentage is {total_percentage}, but it should be 100.")

df_min['angle'] = df_min['Percentage of streaming'] / df_min['Percentage of streaming'].sum() * 2 * pi

custom_colors = ['#FF0000', '#FF3333', '#FF6666', '#FF9999', '#FFCCCC', '#000000']

df_min['color'] = custom_colors

source = ColumnDataSource(df_min)

p = figure(height=350, title="Percentage of Streaming by Platform", toolbar_location=None,
           tools="hover", tooltips="@{Streaming platform}: @angle{0.0%}", x_range=(-0.5, 1.0))

p.wedge(x=0, y=1, radius=0.4,
        start_angle=cumsum('angle', include_zero=True), end_angle=cumsum('angle'),
        line_color="white", fill_color='color', legend_field='Streaming platform', source=source)

p.axis.axis_label = None
p.axis.visible = False
p.grid.grid_line_color = None

output_notebook()
# I was going to export as HTML but finding it tricky to make it work on our PP slide.

show(p)

# 2.2  Library Size Data and Visuals

In [13]:
df_libr

In [14]:
df_libr.info()

In [22]:
#although a small data frame it did need a bit of sorting, so I changed the column to an int and striped out the commas, set the index to help with the chart.
df_libr['Movies'] = df_libr['Movies'].str.replace(',', '').astype(int)
df_libr['TV Shows'] = df_libr['TV Shows'].str.replace(',', '').astype(int)
df_libr.set_index('Streaming platform', inplace=True)
df_libr

In [24]:
# I created a horzontal bar chart from the matplotlib library.
df_sorted = df_libr.sort_values(by='TV Shows', ascending=False)

colors = ['black', 'red',]

df_sorted.plot(kind='barh', color=('red','black'), figsize=(12, 8), width = 0.8)

plt.title('Totals by Category')
plt.xlabel('Totals')
plt.ylabel('Streaming Platforms')
plt.grid(True)

plt.savefig('Platforms_by_streaming_types.png', format='png', dpi=300)




# 2.3  Movies Streams Including IMDb Scoring Data and Visuals

In [28]:
df_movstr.sample(20)


In [29]:
df_movstr.info()

In [38]:
#here I wanted to count the values just in the platforms columns but also keep the IMDb score column. I used a for loop to total the platforms column
#and averaged out the IMDb scores and got it into one small data frame.

str_pltf = ['Netflix', 'Hulu', 'Prime Video', 'Disney+']

results = []

for platform in str_pltf:
    platform_movies = df_movstr[df_movstr[platform] == 1]
    platform_movies = platform_movies.dropna(subset=['IMDb'])
    avg_imdb = platform_movies['IMDb'].mean()
    count = platform_movies.shape[0]
    
    results.append({'Platform': platform, 'Average IMDb': avg_imdb, 'Count': count})

results_df = pd.DataFrame(results)
results_df

In [39]:
results_df.info()

In [61]:
#made an interactive chart on bokeh but then also realized not realistic to link with PP.
source = ColumnDataSource(results_df)

p = figure(x_range=results_df['Platform'], y_range=(4, 8), height=400, width=700,
           title="Average IMDb Rating and Movie Count by Streaming Platform",
           toolbar_location="above", tools="pan,wheel_zoom,box_zoom,reset")

p.vbar(x='Platform', top='Average IMDb', width=0.6, source=source, color='red', legend_label="Average IMDb")

p.extra_y_ranges = {"Count": Range1d(start=0, end=max(results_df['Count']) + 2000)}
p.add_layout(LinearAxis(y_range_name="Count", axis_label="Total movies"), 'right')
p.line(x='Platform', y='Count', source=source, color='black', y_range_name="Count", legend_label="Count", line_width=2)

p.scatter(x='Platform', y='Count', source=source, color='black', y_range_name="Count", size=8, marker='circle')
hover = HoverTool()
hover.tooltips = [
    ("Platform", "@Platform"),
    ("Average IMDb", "@{Average IMDb}{0.2f}"),
    ("Count", "@Count")
]
p.add_tools(hover)
p.yaxis.axis_label = "Average IMDb score "
p.xgrid.grid_line_color = None
p.legend.location = "top_left"
p.legend.title = "Metrics"


show(p)
#some reason I can't make the RH axis read as Total movies

In [63]:
#made a matplotlib chart to show the average IMDb score for the total amount of movies across the competing platforms.

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.bar(results_df.index, results_df['Average IMDb'], color='red', width=0.6, label='Average IMDb')
ax1.set_xlabel('Streaming Platform')
ax1.set_ylabel('Average IMDb Rating', color='black')
ax1.tick_params(axis='y', labelcolor='black')
ax1.set_ylim(4, 8)

ax2 = ax1.twinx()
ax2.plot(results_df.index, results_df['Count'], color='black', marker='o', label='Count')
ax2.set_ylabel('Number of Movies', color='black')
ax2.tick_params(axis='y', labelcolor='black')

plt.title('Average IMDb Rating and Movie Count by Streaming Platform')

fig.tight_layout()

plt.savefig('Avg_IMDb_total_movies.png', format='png', dpi=300)

plt.show()